In [ ]:
# prompt: connect drive

from google.colab import drive
drive.mount('/content/drive')
# ================= INSTALL =================
!pip install flask pyngrok pandas tensorflow pillow rembg onnxruntime paho-mqtt
!pip install pyngrok
!pip install paho-mqtt

from flask import Flask, request, render_template, send_from_directory, redirect, url_for
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from werkzeug.utils import secure_filename
import numpy as np
import os
import requests
import threading
import csv
from datetime import datetime, timedelta
from pyngrok import ngrok
import logging
import time
import paho.mqtt.client as mqtt
import json

# === Pengaturan Lingkungan ===
app_dir = "/content/drive/MyDrive/Sistem Jadi/StrawB"
os.chdir(app_dir) # Hapus tanda # jika kamu jalankan di Google Colab

UPLOAD_FOLDER = 'uploads'
HISTORY_FILE = 'history.csv'

app = Flask(__name__, template_folder='templates', static_folder='static')
app.config['UPLOAD_FOLDER'] = UPLOAD_FOLDER
os.makedirs(UPLOAD_FOLDER, exist_ok=True)

# === Load Model ===
models = {
    "scratch": load_model('models/Salinan model_cnn_strawberry.h5'),
}
class_labels = ["Antraknosa", "Bercak Daun", "Buah Sehat", "Daun Sehat", "Jamur Abu Abu"]

# === Manajemen Riwayat (CSV) ===
def save_history_to_csv(filename=HISTORY_FILE):
    with open(filename, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(['Filename', 'Prediction', 'Confidence', 'Timestamp', 'Action', 'Relay Duration (s)', 'Latency (ms)'])
        for entry in history:
            writer.writerow(entry)

def load_history_from_csv(filename=HISTORY_FILE):
    if os.path.exists(filename):
        with open(filename, mode='r') as file:
            reader = csv.reader(file)
            next(reader, None) # Skip header
            return [row for row in reader]
    return []

history = load_history_from_csv()

# === Fungsi Pembantu ===
def preprocess_image(image_path, target_size=(384, 384)):
    image = load_img(image_path, target_size=target_size)
    image = img_to_array(image)
    image = np.expand_dims(image, axis=0)
    image = image / 255.0
    return image

def kirim_perintah_mqtt(klasifikasi, durasi=10):
    broker = "2c1a605f76c94605924c519758e30406.s1.eu.hivemq.cloud"
    port = 8883
    username = "tugasyoshi"
    password = "Cobalagi1122"

    topic_mapping = {
        "Antraknosa": "plant/relay/1",
        "Bercak Daun": "plant/relay/2",
        "Jamur Abu Abu": "plant/relay/3"
    }

    topic = topic_mapping.get(klasifikasi)
    if not topic:
        return

    payload = {"state": "ON", "duration": durasi}
    client = mqtt.Client(client_id=f"flask-client-{int(time.time())}")
    client.username_pw_set(username, password)
    client.tls_set()
    client.tls_insecure_set(True)

    try:
        client.connect(broker, port)
        client.loop_start()
        client.publish(topic, json.dumps(payload))
        time.sleep(1)
        client.loop_stop()
        client.disconnect()
        app.logger.info(f"[MQTT] Terkirim ke {topic} -> {payload}")
    except Exception as e:
        app.logger.error(f"[MQTT] Gagal kirim ke {topic}: {e}")

def format_duration(seconds):
    seconds = int(seconds)
    h, r = divmod(seconds, 3600)
    m, s = divmod(r, 60)
    return ' '.join(f"{v} {u}" for v, u in zip((h, m, s), ('jam', 'menit', 'detik')) if v)

# === Rute Aplikasi (Routing) ===
@app.route('/uploads/<filename>')
def uploaded_file(filename):
    return send_from_directory(app.config['UPLOAD_FOLDER'], filename)

@app.route('/', methods=['GET', 'POST'])
def predict():
    uploaded_image_path = None
    duration = 15 # Default 15 detik
    formatted_duration = format_duration(duration)
    action = None
    latency_ms = None
    detected_disease = None
    confidence = 0

    if request.method == 'POST':
        if 'image' not in request.files or request.files['image'].filename == '':
            return render_template('index.html', error="Silakan unggah gambar.", duration=duration)

        file = request.files['image']
        filename = secure_filename(file.filename)
        filepath = os.path.join(app.config['UPLOAD_FOLDER'], filename)

        # Cegah timpa file dengan nama sama
        if os.path.exists(filepath):
            timestamp_file = int(time.time())
            filename = f"{timestamp_file}_{filename}"
            filepath = os.path.join(app.config['UPLOAD_FOLDER'], filename)

        file.save(filepath)
        uploaded_image_path = filename

        # Ambil durasi dari form
        try:
            duration = int(request.form.get('duration', 15))
        except ValueError:
            duration = 15

        formatted_duration = format_duration(duration)

        # Inferensi model
        image = preprocess_image(filepath)
        pred_array = models["scratch"].predict(image)[0]
        max_idx = np.argmax(pred_array)
        detected_disease = class_labels[max_idx]
        confidence = round(float(pred_array[max_idx]) * 100, 1)

        relay_duration = 0

        # Logika Aksi & IoT
        if detected_disease in ["Antraknosa", "Jamur Abu Abu", "Bercak Daun"]:
            action = f"Penyakit {detected_disease} terdeteksi. Sistem mengaktifkan penyemprotan pestisida selama {formatted_duration}."
            relay_duration = duration
            t0 = time.time()
            kirim_perintah_mqtt(detected_disease, duration)
            t1 = time.time()
            latency_ms = round((t1 - t0) * 1000, 2)
        else:
            action = "Buah atau daun sehat. Tidak ada tindakan."
            latency_ms = 0

        timestamp_log = (datetime.utcnow() + timedelta(hours=7)).strftime('%Y-%m-%d %H:%M:%S')
        
        # Simpan History di indeks paling atas (terbaru)
        history.insert(0, [filename, detected_disease, confidence, timestamp_log, action, relay_duration, latency_ms])
        save_history_to_csv()

    return render_template('index.html',
                           uploaded_image_path=uploaded_image_path,
                           detected_disease=detected_disease,
                           confidence=confidence,
                           duration=duration,
                           action=action,
                           latency=latency_ms)

@app.route('/history')
def show_history():
    return render_template('history.html', history=history)

@app.route('/save-history')
def save_history():
    save_history_to_csv()
    return send_from_directory(directory='.', path=HISTORY_FILE, as_attachment=True)

@app.route('/delete-image', methods=['POST'])
def delete_image():
    image_path = request.form.get('image_path')
    full_path = os.path.join(app.config['UPLOAD_FOLDER'], image_path)
    if os.path.exists(full_path):
        os.remove(full_path)

    global history
    history = [entry for entry in history if entry[0] != image_path]
    save_history_to_csv()
    return redirect(url_for('show_history'))

@app.route('/delete-all', methods=['POST'])
def delete_all_history():
    with open(HISTORY_FILE, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['Filename', 'Prediction', 'Confidence', 'Timestamp', 'Action', 'Relay Duration (s)', 'Latency (ms)'])

    for f in os.listdir(UPLOAD_FOLDER):
        os.remove(os.path.join(UPLOAD_FOLDER, f))

    global history
    history = []
    return redirect(url_for('show_history'))

# === Jalankan Flask + Ngrok (Versi Diperbaiki) ===
def run_app():
    # Gunakan debug=False agar log tidak menelan output URL Ngrok
    app.run(port=5000, debug=False, use_reloader=False)

if __name__ == '__main__':
    app.logger.setLevel(logging.INFO)
    os.makedirs(UPLOAD_FOLDER, exist_ok=True)
    
    # 1. Matikan sesi ngrok yang mungkin masih nyangkut
    ngrok.kill()

    # 2. Setup Auth Ngrok
    ngrok.set_auth_token("2xGoN4jsFB5UYrHf7u9T2BSAF3E_4WpuepbPsxGAixfH6M7Nm")
    
    # 3. Buat tunnel baru
    tunnel = ngrok.connect(5000)
    public_url = tunnel.public_url

    # 4. Cetak URL dengan format mencolok
    print("\n" + "="*65)
    print(f"🚀 BUKA LINK INI UNTUK MENGAKSES WEB: {public_url}")
    print("="*65 + "\n")

    # 5. Jalankan Flask di Thread
    flask_thread = threading.Thread(target=run_app)
    flask_thread.start()